# NSE Daily Stocks + NIFTY 50 — Fast Incremental Sync

This Colab notebook synchronizes:

- NSE Cash Market daily stock Parquet files
- NIFTY 50 daily OHLC Parquet files
- Missing dates only
- No full Parquet validation of existing stock files during the normal run
- Atomic Parquet writes
- CSV manifests
- Progress, rate, ETA and error reporting
- DuckDB sanity checks

## Important performance behavior

The stock sync is intentionally split into two phases:

1. **Fast inventory:** checks only whether the expected Parquet file exists.
2. **Download:** processes **only missing dates**.

For example:

```text
Weekday candidates : 4,100
Existing files     : 3,332
Missing files      : 768
```

The download loop will therefore show:

```text
[1/768]
[25/768]
...
[768/768]
```

It will **not** loop through 4,100 existing dates.

Existing files are not opened/read during the normal stock inventory.

> Optional deep validation is provided separately for sampled existing files.

In [ ]:
!pip -q install pandas pyarrow requests duckdb

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import date, datetime, timedelta
import io
import json
import time
import zipfile
import logging
import math

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import duckdb

print("Environment ready.")

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/quant")
DATA_DIR = BASE_DIR / "data"

PARQUET_DIR = DATA_DIR / "parquet"
RAW_DIR = DATA_DIR / "raw" / "nse_bhavcopy"
METADATA_DIR = DATA_DIR / "metadata"

INDEX_DATA_DIR = DATA_DIR / "indices"
NIFTY50_DIR = INDEX_DATA_DIR / "nifty50"

MANIFEST_FILE = DATA_DIR / "download_manifest.csv"
NIFTY50_MANIFEST_FILE = INDEX_DATA_DIR / "nifty50_manifest.csv"

# Historical range
START_DATE = date(2011, 1, 1)
END_DATE = None   # None = today

# NSE stock format transition
LEGACY_END_DATE = date(2024, 7, 5)

# Optional stock filter
# None = all securities
# Example: {"RELIANCE", "TCS", "INFY"}
STOCKS = None

# Network
REQUEST_TIMEOUT = 60
MAX_RETRIES = 5
BACKOFF_FACTOR = 1.5
SLEEP_BETWEEN_REQUESTS = 0.10

# Existing files are NOT deeply validated during normal inventory.
# This is intentional for speed on Google Drive.
DEEP_VALIDATE_EXISTING = False
DEEP_VALIDATE_SAMPLE_SIZE = 100

# Repair newly discovered invalid files only when deep validation is enabled.
REPAIR_INVALID_FILES = True

# Raw ZIP retention
KEEP_RAW_ZIPS = False

# Progress
PROGRESS_EVERY = 25
PROGRESS_EVERY_SECONDS = 20

EXPECTED_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

if END_DATE is None:
    END_DATE = date.today()

for directory in [
    BASE_DIR,
    DATA_DIR,
    PARQUET_DIR,
    RAW_DIR,
    METADATA_DIR,
    INDEX_DATA_DIR,
    NIFTY50_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

assert START_DATE <= END_DATE

print("BASE_DIR :", BASE_DIR)
print("DATA_DIR :", DATA_DIR)
print("STOCK    :", PARQUET_DIR)
print("NIFTY50  :", NIFTY50_DIR)
print("RANGE    :", START_DATE, "to", END_DATE)

In [ ]:
# ============================================================
# 3. LOGGING + HTTP SESSION
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger("nse_sync")

session = requests.Session()

retry = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=BACKOFF_FACTOR,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET", "POST"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
})

print("HTTP session configured.")

In [ ]:
# ============================================================
# 4. DATE / PATH HELPERS
# ============================================================

def trading_day_candidates(start_date: date, end_date: date):
    current = start_date
    while current <= end_date:
        if current.weekday() < 5:
            yield current
        current += timedelta(days=1)


def parquet_path_for_day(day: date) -> Path:
    year_dir = PARQUET_DIR / f"year={day.year}"
    return year_dir / f"nse_cm_{day:%Y%m%d}.parquet"


def nifty50_parquet_path_for_day(day: date) -> Path:
    year_dir = NIFTY50_DIR / f"year={day.year}"
    return year_dir / f"nifty50_{day:%Y%m%d}.parquet"


def raw_zip_path_for_day(day: date) -> Path:
    return RAW_DIR / f"nse_cm_{day:%Y%m%d}.zip"


print("Path helpers ready.")

In [ ]:
# ============================================================
# 5. NSE STOCK URL HELPERS
# ============================================================

def legacy_url(day: date) -> str:
    month = day.strftime("%b").upper()
    filename = f"cm{day:%d}{month}{day:%Y}bhav.csv.zip"

    return (
        "https://nsearchives.nseindia.com/content/"
        f"historical/EQUITIES/{day.year}/{month}/{filename}"
    )


def udiff_url(day: date) -> str:
    filename = (
        f"BhavCopy_NSE_CM_0_0_0_"
        f"{day:%Y%m%d}_F_0000.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/cm/"
        f"{filename}"
    )


def url_for_day(day: date) -> str:
    if day <= LEGACY_END_DATE:
        return legacy_url(day)
    return udiff_url(day)


for d in [
    date(2011, 1, 3),
    date(2024, 7, 5),
    date(2024, 7, 8),
]:
    print(d, "->", url_for_day(d))

In [ ]:
# ============================================================
# 6. STOCK NORMALIZATION
# ============================================================

def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}

    aliases = {
        "tradingsymbol": "symbol",
        "symbol": "symbol",

        "timestamp": "date",
        "trade_date": "date",
        "date": "date",

        "open_price": "open",
        "open": "open",

        "high_price": "high",
        "high": "high",

        "low_price": "low",
        "low": "low",

        "close_price": "close",
        "close": "close",

        "last_price": "close",
        "ltp": "close",

        "tottrdqty": "volume",
        "total_traded_quantity": "volume",
        "total_traded_qty": "volume",
        "volume": "volume",
    }

    for col in df.columns:
        normalized = normalize_column_name(col)
        if normalized in aliases:
            rename[col] = aliases[normalized]

    df = df.rename(columns=rename)

    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Received columns: {list(df.columns)}"
        )

    df = df[EXPECTED_COLUMNS].copy()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce",
    ).dt.date

    df["symbol"] = (
        df["symbol"]
        .astype("string")
        .str.strip()
    )

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date", "symbol"])

    if STOCKS is not None:
        df = df[df["symbol"].isin(STOCKS)]

    df = df.drop_duplicates(
        subset=["date", "symbol"],
        keep="last",
    )

    return df


def read_nse_zip(content: bytes) -> pd.DataFrame:
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        csv_names = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_names:
            raise ValueError("ZIP archive contains no CSV.")

        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f)

    return normalize_columns(df)

In [ ]:
# ============================================================
# 7. STOCK VALIDATION + ATOMIC PARQUET WRITE
# ============================================================

def validate_dataframe(
    df: pd.DataFrame,
    expected_day: date,
) -> tuple[bool, str]:

    if df.empty:
        return False, "empty_dataframe"

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        return False, f"missing_columns:{missing}"

    if df["date"].isna().any():
        return False, "null_dates"

    if not (df["date"] == expected_day).all():
        return False, "wrong_date"

    if df["symbol"].isna().any():
        return False, "invalid_symbols"

    numeric_cols = ["open", "high", "low", "close", "volume"]

    if df[numeric_cols].isna().all(axis=1).any():
        return False, "rows_with_all_numeric_values_null"

    return True, "ok"


def validate_parquet(
    path: Path,
    expected_day: date,
) -> tuple[bool, str, int]:

    if not path.exists():
        return False, "missing_file", 0

    try:
        table = pq.read_table(path)
        df = normalize_columns(table.to_pandas())

        ok, reason = validate_dataframe(
            df,
            expected_day,
        )

        return ok, reason, len(df)

    except Exception as exc:
        return (
            False,
            f"parquet_read_error:{type(exc).__name__}:{exc}",
            0,
        )


def atomic_write_parquet(
    df: pd.DataFrame,
    path: Path,
):
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(".parquet.tmp")
    tmp_path.unlink(missing_ok=True)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False,
    )

    pq.write_table(
        table,
        tmp_path,
        compression="snappy",
    )

    ok, reason, rows = validate_parquet(
        tmp_path,
        df["date"].iloc[0],
    )

    if not ok:
        tmp_path.unlink(missing_ok=True)
        raise ValueError(
            f"Post-write validation failed: {reason}"
        )

    tmp_path.replace(path)

    return rows

In [ ]:
# ============================================================
# 8. STOCK DOWNLOAD
# ============================================================

def download_stock_day(day: date) -> dict:

    path = parquet_path_for_day(day)
    url = url_for_day(day)

    # Safety check only.
    # Normal inventory ensures existing files never enter this function.
    # If a file appears between inventory and download, do not overwrite it.
    if path.exists():
        return {
            "date": day.isoformat(),
            "status": "already_exists",
            "rows": 0,
            "path": str(path),
            "url": url,
            "message": "File appeared after inventory scan.",
        }

    try:
        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
        )

        if response.status_code == 404:
            return {
                "date": day.isoformat(),
                "status": "not_found",
                "rows": 0,
                "path": str(path),
                "url": url,
                "message": (
                    "NSE archive not found; "
                    "likely holiday/non-trading day."
                ),
            }

        response.raise_for_status()

        content = response.content

        if not content:
            raise ValueError("NSE returned an empty response.")

        if KEEP_RAW_ZIPS:
            raw_zip_path_for_day(day).write_bytes(content)

        df = read_nse_zip(content)

        ok, reason = validate_dataframe(
            df,
            day,
        )

        if not ok:
            raise ValueError(
                f"Downloaded data failed validation: {reason}"
            )

        rows = atomic_write_parquet(
            df,
            path,
        )

        return {
            "date": day.isoformat(),
            "status": "downloaded",
            "rows": rows,
            "path": str(path),
            "url": url,
            "message": "Downloaded successfully.",
        }

    except Exception as exc:
        return {
            "date": day.isoformat(),
            "status": "error",
            "rows": 0,
            "path": str(path),
            "url": url,
            "message": f"{type(exc).__name__}: {exc}",
        }

In [ ]:
# ============================================================
# 9. STOCK MANIFEST + RESUMABLE SYNC HELPERS
# ============================================================

MANIFEST_COLUMNS = [
    "date", "status", "rows", "path", "url", "message", "checked_at",
]

def load_manifest() -> pd.DataFrame:
    if not MANIFEST_FILE.exists():
        return pd.DataFrame(columns=MANIFEST_COLUMNS)
    try:
        manifest = pd.read_csv(MANIFEST_FILE)
        for col in MANIFEST_COLUMNS:
            if col not in manifest.columns:
                manifest[col] = None
        manifest["date"] = manifest["date"].astype(str)
        return manifest[MANIFEST_COLUMNS]
    except Exception as exc:
        logger.warning("Could not read manifest: %s", exc)
        return pd.DataFrame(columns=MANIFEST_COLUMNS)

manifest = load_manifest()

def append_manifest(record: dict):
    global manifest
    row = {col: record.get(col) for col in MANIFEST_COLUMNS}
    row["checked_at"] = datetime.now().isoformat(timespec="seconds")

    manifest = pd.concat(
        [manifest, pd.DataFrame([row])],
        ignore_index=True,
    )
    manifest["date"] = manifest["date"].astype(str)
    manifest = (
        manifest
        .drop_duplicates(subset=["date"], keep="last")
        .sort_values("date")
    )
    manifest.to_csv(MANIFEST_FILE, index=False)

def manifest_status_sets(manifest_df: pd.DataFrame):
    """Return latest manifest status sets by date."""
    if manifest_df.empty:
        return set(), set(), set()

    latest = (
        manifest_df.copy()
        .assign(
            date=lambda x: x["date"].astype(str),
            checked_at=lambda x: x["checked_at"].fillna(""),
        )
        .sort_values(["date", "checked_at"])
        .drop_duplicates(subset=["date"], keep="last")
    )

    completed = set(
        latest.loc[latest["status"].eq("downloaded"), "date"]
    )
    not_found = set(
        latest.loc[latest["status"].eq("not_found"), "date"]
    )
    failed = set(
        latest.loc[latest["status"].eq("error"), "date"]
    )

    return completed, not_found, failed

print("Manifest:", MANIFEST_FILE)
print("Manifest records:", f"{len(manifest):,}")


In [ ]:
# ============================================================
# 10. RESUMABLE STOCK INVENTORY
# ============================================================

inventory_started = datetime.now()

candidates = list(
    trading_day_candidates(START_DATE, END_DATE)
)

candidate_iso = {day.isoformat() for day in candidates}

manifest_completed, manifest_not_found, manifest_failed = (
    manifest_status_sets(manifest)
)

completed_in_range = candidate_iso & manifest_completed
not_found_in_range = candidate_iso & manifest_not_found
failed_in_range = candidate_iso & manifest_failed

# Dates already marked downloaded/not_found are skipped.
# Failed dates are intentionally retried.
manifest_skipped = completed_in_range | not_found_in_range

pending_from_manifest = [
    day for day in candidates
    if day.isoformat() not in manifest_skipped
]

# Filesystem check is only for dates not already terminal in the manifest.
# Existing files are skipped without opening the Parquet.
stock_existing = []
stock_missing = []

for day in pending_from_manifest:
    path = parquet_path_for_day(day)
    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)

inventory_finished = datetime.now()

print("=" * 70)
print("RESUMABLE STOCK SYNC CHECK")
print("=" * 70)
print("Weekday candidates       :", f"{len(candidates):,}")
print("Manifest completed       :", f"{len(completed_in_range):,}")
print("Manifest not found       :", f"{len(not_found_in_range):,}")
print("Manifest failed          :", f"{len(failed_in_range):,}")
print("Existing file, no action :", f"{len(stock_existing):,}")
print("Pending / to download    :", f"{len(stock_missing):,}")
print("Inventory elapsed        :", inventory_finished - inventory_started)

resume_summary = pd.DataFrame([
    {"status": "pending", "count": len(stock_missing)},
    {"status": "completed", "count": len(completed_in_range)},
    {"status": "not_found", "count": len(not_found_in_range)},
    {"status": "failed", "count": len(failed_in_range)},
    {"status": "existing_file", "count": len(stock_existing)},
])
display(resume_summary)

if stock_missing:
    print("First pending date:", stock_missing[0])
    print("Last pending date :", stock_missing[-1])

print()
print("Download loop will process ONLY:", f"{len(stock_missing):,}", "pending dates.")


In [ ]:
# ============================================================
# 11. OPTIONAL DEEP VALIDATION OF EXISTING FILES
# ============================================================
# Disabled by default.
#
# Use this only when you explicitly want to audit existing files.
# It reads Parquet files and is therefore much slower on Google Drive.

def deep_validate_existing_files(
    days,
    sample_size=100,
):
    if not days:
        print("No existing files to validate.")
        return pd.DataFrame()

    sample = list(days[:sample_size])

    results = []

    print(
        f"Deep-validating {len(sample):,} existing files..."
    )

    for i, day in enumerate(sample, 1):

        path = parquet_path_for_day(day)

        ok, reason, rows = validate_parquet(
            path,
            day,
        )

        results.append({
            "date": day.isoformat(),
            "valid": ok,
            "reason": reason,
            "rows": rows,
            "path": str(path),
        })

        if i % 25 == 0 or i == len(sample):
            print(
                f"[{i:,}/{len(sample):,}] "
                f"{i / len(sample) * 100:.1f}%"
            )

    result_df = pd.DataFrame(results)

    print()
    print("Deep validation summary:")
    display(
        result_df["valid"]
        .value_counts()
        .rename_axis("valid")
        .reset_index(name="count")
    )

    invalid = result_df[~result_df["valid"]]

    if not invalid.empty:
        print("Invalid files found:")
        display(invalid)

    return result_df


if DEEP_VALIDATE_EXISTING:
    deep_validation_df = deep_validate_existing_files(
        stock_existing,
        DEEP_VALIDATE_SAMPLE_SIZE,
    )
else:
    print(
        "Deep validation disabled. "
        "Existing files are trusted by filesystem presence."
    )

In [ ]:
# ============================================================
# 12. RESUMABLE STOCK DOWNLOAD LOOP
# ============================================================

run_started = datetime.now()
run_results = []

total_pending = len(stock_missing)

print()
print("=" * 70)
print("RESUMABLE STOCK DOWNLOAD")
print("=" * 70)
print("Pending dates to process          :", f"{total_pending:,}")
print("Manifest-downloaded dates skipped :", f"{len(completed_in_range):,}")
print("Manifest-not-found dates skipped  :", f"{len(not_found_in_range):,}")
print("Failed dates being retried        :", f"{len(failed_in_range):,}")
print("Existing files skipped            :", f"{len(stock_existing):,}")
print()

if total_pending == 0:
    print("Nothing pending.")
else:
    last_progress_time = time.monotonic()

    for i, day in enumerate(stock_missing, 1):

        result = download_stock_day(day)
        run_results.append(result)
        append_manifest(result)

        now = time.monotonic()
        should_report = (
            i == 1
            or i == total_pending
            or i % PROGRESS_EVERY == 0
            or (now - last_progress_time >= PROGRESS_EVERY_SECONDS)
        )

        if should_report:
            elapsed = (datetime.now() - run_started).total_seconds()
            rate = i / elapsed if elapsed > 0 else 0
            remaining = total_pending - i
            eta_seconds = remaining / rate if rate > 0 else 0
            pct = i / total_pending * 100

            downloaded = sum(r["status"] == "downloaded" for r in run_results)
            not_found = sum(r["status"] == "not_found" for r in run_results)
            errors = sum(r["status"] == "error" for r in run_results)
            already_exists = sum(
                r["status"] == "already_exists" for r in run_results
            )

            print(
                f"[{i:,}/{total_pending:,}] "
                f"{pct:6.2f}% | Date: {day} | "
                f"Rate: {rate:.2f}/sec | "
                f"ETA: {timedelta(seconds=int(eta_seconds))} | "
                f"Downloaded: {downloaded:,} | "
                f"404: {not_found:,} | "
                f"Race-skip: {already_exists:,} | "
                f"Errors: {errors:,}"
            )
            last_progress_time = now

        if result["status"] in {"downloaded", "error"}:
            if SLEEP_BETWEEN_REQUESTS:
                time.sleep(SLEEP_BETWEEN_REQUESTS)

run_finished = datetime.now()
run_df = pd.DataFrame(run_results)

print()
print("=" * 70)
print("RESUMABLE STOCK SYNC COMPLETED")
print("=" * 70)
print("Started :", run_started)
print("Finished:", run_finished)
print("Elapsed :", run_finished - run_started)
print("Processed:", f"{len(run_results):,}", "/", f"{total_pending:,}")

if not run_df.empty:
    display(
        run_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )


In [ ]:
# ============================================================
# 13. STOCK SYNC SUMMARY
# ============================================================

if run_df.empty:

    print("No stock dates processed.")

else:

    print("Stock status counts:")

    display(
        run_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    downloaded_rows = run_df.loc[
        run_df["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Rows downloaded:",
        f"{downloaded_rows:,}",
    )

    errors = run_df[
        run_df["status"] == "error"
    ]

    if not errors.empty:
        print("Stock errors:")
        display(
            errors[
                [
                    "date",
                    "message",
                    "url",
                ]
            ]
        )

## NIFTY 50

The NSE `indicesHistory` endpoint can sometimes return HTML instead of JSON.

This notebook therefore uses the Nifty Indices historical-data endpoint for NIFTY 50:

`https://niftyindices.com/Backpage.aspx/getHistoricaldatatabletoString`

The response is normalized into the same schema:

`date, symbol, open, high, low, close, volume`

with:

`symbol = NIFTY50`

and:

`volume = null`.

In [ ]:
# ============================================================
# 14. NIFTY 50 SOURCE
# ============================================================

NIFTY_SOURCE_URL = (
    "https://niftyindices.com/Backpage.aspx/"
    "getHistoricaldatatabletoString"
)

NIFTY_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json; charset=UTF-8",
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/",
    "X-Requested-With": "XMLHttpRequest",
}


def nifty50_request(
    start_date: date,
    end_date: date,
):

    cinfo = {
        "name": "NIFTY 50",
        "startDate": start_date.strftime("%d-%b-%Y"),
        "endDate": end_date.strftime("%d-%b-%Y"),
        "indexName": "NIFTY 50",
    }

    payload = {
        "cinfo": json.dumps(cinfo)
    }

    response = session.post(
        NIFTY_SOURCE_URL,
        headers=NIFTY_HEADERS,
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )

    response.raise_for_status()

    return response


def parse_nifty50_response(response) -> pd.DataFrame:

    payload = response.json()

    raw = payload.get("d")

    if raw is None:
        raise ValueError(
            f"NIFTY response missing 'd': {payload}"
        )

    # Nifty Indices commonly returns JSON encoded inside "d".
    if isinstance(raw, str):
        raw = json.loads(raw)

    if isinstance(raw, dict):
        records = (
            raw.get("data")
            or raw.get("records")
            or raw.get("aaData")
            or []
        )
    elif isinstance(raw, list):
        records = raw
    else:
        records = []

    if not records:
        return pd.DataFrame(columns=EXPECTED_COLUMNS)

    rows = []

    for record in records:

        if not isinstance(record, dict):
            continue

        raw_date = (
            record.get("HistoricalDate")
            or record.get("Date")
            or record.get("date")
        )

        parsed_date = pd.to_datetime(
            raw_date,
            errors="coerce",
            dayfirst=True,
        )

        if pd.isna(parsed_date):
            continue

        rows.append({
            "date": parsed_date.date(),
            "symbol": "NIFTY50",
            "open": record.get("OPEN"),
            "high": record.get("HIGH"),
            "low": record.get("LOW"),
            "close": record.get("CLOSE"),
            "volume": None,
        })

    if not rows:
        return pd.DataFrame(columns=EXPECTED_COLUMNS)

    df = pd.DataFrame(rows)

    # NIFTY-specific normalization
    for col in ["open", "high", "low", "close"]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    df["volume"] = None

    return df[
        EXPECTED_COLUMNS
    ].drop_duplicates(
        subset=["date", "symbol"],
        keep="last",
    )


print("NIFTY source:", NIFTY_SOURCE_URL)

In [ ]:
# ============================================================
# 15. NIFTY 50 VALIDATION + WRITE
# ============================================================

def validate_nifty50_dataframe(
    df: pd.DataFrame,
    expected_day: date,
) -> tuple[bool, str]:

    if df.empty:
        return False, "empty_dataframe"

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        return False, f"missing_columns:{missing}"

    if not (df["symbol"] == "NIFTY50").all():
        return False, "unexpected_symbol"

    if not (df["date"] == expected_day).all():
        return False, "wrong_date"

    price_columns = [
        "open",
        "high",
        "low",
        "close",
    ]

    if df[price_columns].isna().any().any():
        return False, "missing_ohlc"

    if len(df) != 1:
        return False, f"expected_one_row_got_{len(df)}"

    return True, "ok"


def validate_nifty50_parquet(
    path: Path,
    expected_day: date,
) -> tuple[bool, str, int]:

    if not path.exists():
        return False, "missing_file", 0

    try:
        table = pq.read_table(path)
        df = table.to_pandas()

        # NIFTY files have a fixed schema.
        for col in EXPECTED_COLUMNS:
            if col not in df.columns:
                return False, f"missing_columns:{col}", len(df)

        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce",
        ).dt.date

        ok, reason = validate_nifty50_dataframe(
            df,
            expected_day,
        )

        return ok, reason, len(df)

    except Exception as exc:
        return (
            False,
            f"parquet_read_error:{type(exc).__name__}:{exc}",
            0,
        )


def write_nifty50_day(
    df: pd.DataFrame,
    path: Path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_path = path.with_suffix(".parquet.tmp")
    tmp_path.unlink(missing_ok=True)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False,
    )

    pq.write_table(
        table,
        tmp_path,
        compression="snappy",
    )

    ok, reason, rows = validate_nifty50_parquet(
        tmp_path,
        df["date"].iloc[0],
    )

    if not ok:
        tmp_path.unlink(missing_ok=True)
        raise ValueError(
            f"NIFTY post-write validation failed: {reason}"
        )

    tmp_path.replace(path)

    return rows

In [ ]:
# ============================================================
# 16. NIFTY 50 FAST INVENTORY
# ============================================================

nifty_candidates = list(
    trading_day_candidates(
        START_DATE,
        END_DATE,
    )
)

nifty_missing = []
nifty_existing = []

for day in nifty_candidates:

    path = nifty50_parquet_path_for_day(day)

    if path.exists():
        nifty_existing.append(day)
    else:
        nifty_missing.append(day)

print("=" * 70)
print("FAST NIFTY 50 INVENTORY")
print("=" * 70)

print(
    "Weekday candidates :",
    f"{len(nifty_candidates):,}",
)

print(
    "Existing files     :",
    f"{len(nifty_existing):,}",
)

print(
    "Missing files      :",
    f"{len(nifty_missing):,}",
)

coverage = (
    len(nifty_existing) / len(nifty_candidates) * 100
    if nifty_candidates else 0
)

print(
    "Coverage           :",
    f"{coverage:.2f}%",
)

if nifty_missing:
    print("First missing date:", nifty_missing[0])
    print("Last missing date :", nifty_missing[-1])
else:
    print("No NIFTY 50 dates need downloading.")

In [ ]:
# ============================================================
# 17. NIFTY 50 SYNC
# ============================================================

def chunked_dates(days, chunk_size=90):
    for i in range(0, len(days), chunk_size):
        yield days[i:i + chunk_size]


def sync_nifty50_missing(
    missing_dates,
    chunk_size=90,
):

    results = []

    total_missing = len(missing_dates)

    print()
    print("=" * 70)
    print("NIFTY 50 DOWNLOAD")
    print("=" * 70)
    print(
        f"Dates to download: {total_missing:,}"
    )

    if total_missing == 0:
        print("Nothing to download.")
        return pd.DataFrame()

    started = datetime.now()

    chunks = list(
        chunked_dates(
            missing_dates,
            chunk_size,
        )
    )

    for chunk_no, chunk in enumerate(
        chunks,
        1,
    ):

        chunk_start = chunk[0]
        chunk_end = chunk[-1]

        print()
        print(
            f"NIFTY chunk "
            f"{chunk_no}/{len(chunks)}: "
            f"{chunk_start} -> {chunk_end} "
            f"({len(chunk):,} missing dates)"
        )

        request_started = time.monotonic()

        try:

            response = nifty50_request(
                chunk_start,
                chunk_end,
            )

            request_seconds = (
                time.monotonic()
                - request_started
            )

            print(
                f"HTTP {response.status_code} | "
                f"request: {request_seconds:.2f}s"
            )

            df = parse_nifty50_response(
                response
            )

            print(
                f"Returned NIFTY records: {len(df):,}"
            )

            returned_dates = set(
                df["date"].tolist()
            ) if not df.empty else set()

            for day in chunk:

                path = nifty50_parquet_path_for_day(
                    day
                )

                # Race protection
                if path.exists():
                    results.append({
                        "date": day.isoformat(),
                        "status": "already_exists",
                        "rows": 0,
                        "path": str(path),
                        "message": (
                            "File appeared after inventory."
                        ),
                    })
                    continue

                day_df = df[
                    df["date"] == day
                ].copy()

                if day_df.empty:

                    results.append({
                        "date": day.isoformat(),
                        "status": "not_found",
                        "rows": 0,
                        "path": str(path),
                        "message": (
                            "No NIFTY50 observation returned."
                        ),
                    })

                    continue

                ok, reason = validate_nifty50_dataframe(
                    day_df,
                    day,
                )

                if not ok:

                    results.append({
                        "date": day.isoformat(),
                        "status": "error",
                        "rows": 0,
                        "path": str(path),
                        "message": reason,
                    })

                    continue

                rows = write_nifty50_day(
                    day_df,
                    path,
                )

                results.append({
                    "date": day.isoformat(),
                    "status": "downloaded",
                    "rows": rows,
                    "path": str(path),
                    "message": "Downloaded successfully.",
                })

            processed = len(results)

            elapsed = (
                datetime.now() - started
            ).total_seconds()

            rate = (
                processed / elapsed
                if elapsed > 0
                else 0
            )

            remaining = total_missing - processed

            eta_seconds = (
                remaining / rate
                if rate > 0
                else 0
            )

            print(
                f"Progress: {processed:,}/"
                f"{total_missing:,} "
                f"({processed / total_missing * 100:.1f}%) | "
                f"Rate: {rate:.2f}/sec | "
                f"ETA: "
                f"{timedelta(seconds=int(eta_seconds))}"
            )

        except Exception as exc:

            print(
                f"NIFTY request failed: "
                f"{type(exc).__name__}: {exc}"
            )

            for day in chunk:

                path = nifty50_parquet_path_for_day(
                    day
                )

                results.append({
                    "date": day.isoformat(),
                    "status": "error",
                    "rows": 0,
                    "path": str(path),
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                })

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    return pd.DataFrame(results)


nifty50_started = datetime.now()

nifty50_results = sync_nifty50_missing(
    nifty_missing,
    chunk_size=90,
)

nifty50_finished = datetime.now()

print()
print("=" * 70)
print("NIFTY 50 SYNC COMPLETE")
print("=" * 70)
print("Started :", nifty50_started)
print("Finished:", nifty50_finished)
print("Elapsed :", nifty50_finished - nifty50_started)

In [ ]:
# ============================================================
# 18. NIFTY 50 MANIFEST
# ============================================================

def save_nifty50_manifest(df: pd.DataFrame):

    if df.empty:
        return

    manifest = df.copy()

    manifest["checked_at"] = (
        datetime.now().isoformat(
            timespec="seconds"
        )
    )

    if NIFTY50_MANIFEST_FILE.exists():

        old = pd.read_csv(
            NIFTY50_MANIFEST_FILE
        )

        manifest = pd.concat(
            [old, manifest],
            ignore_index=True,
        )

    manifest["date"] = manifest["date"].astype(str)

    manifest = (
        manifest
        .drop_duplicates(
            subset=["date"],
            keep="last",
        )
        .sort_values("date")
    )

    manifest.to_csv(
        NIFTY50_MANIFEST_FILE,
        index=False,
    )


save_nifty50_manifest(
    nifty50_results
)

if not nifty50_results.empty:

    print("NIFTY 50 status:")

    display(
        nifty50_results["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="count")
    )

    errors = nifty50_results[
        nifty50_results["status"] == "error"
    ]

    if not errors.empty:
        print("NIFTY 50 errors:")
        display(
            errors[
                [
                    "date",
                    "message",
                    "path",
                ]
            ]
        )

print("NIFTY50 directory:", NIFTY50_DIR)
print("NIFTY50 manifest :", NIFTY50_MANIFEST_FILE)

In [ ]:
# ============================================================
# 19. FINAL FAST FILE INVENTORY
# ============================================================
# Uses Parquet metadata only. It does not load the datasets
# into pandas.

def parquet_inventory(directory: Path):

    files = sorted(
        directory.glob(
            "year=*/**/*.parquet"
        )
    )

    rows = []

    for path in files:

        try:

            metadata = pq.ParquetFile(
                path
            ).metadata

            rows.append({
                "file": str(path),
                "rows": metadata.num_rows,
                "size_mb": (
                    path.stat().st_size
                    / (1024 ** 2)
                ),
            })

        except Exception as exc:

            rows.append({
                "file": str(path),
                "rows": None,
                "size_mb": None,
                "error": str(exc),
            })

    return pd.DataFrame(rows)


stock_inventory = parquet_inventory(
    PARQUET_DIR
)

nifty_inventory = parquet_inventory(
    NIFTY50_DIR
)

print("STOCK DATA")
print("-" * 50)
print("Files:", len(stock_inventory))
print(
    "Rows:",
    f"{stock_inventory['rows'].sum():,.0f}"
    if not stock_inventory.empty
    else 0,
)
print(
    "Size MB:",
    f"{stock_inventory['size_mb'].sum():,.1f}"
    if not stock_inventory.empty
    else 0,
)

print()
print("NIFTY 50 DATA")
print("-" * 50)
print("Files:", len(nifty_inventory))
print(
    "Rows:",
    f"{nifty_inventory['rows'].sum():,.0f}"
    if not nifty_inventory.empty
    else 0,
)
print(
    "Size MB:",
    f"{nifty_inventory['size_mb'].sum():,.1f}"
    if not nifty_inventory.empty
    else 0,
)

In [ ]:
# ============================================================
# 20. DUCKDB SANITY CHECK
# ============================================================

con = duckdb.connect()

stock_files = list(
    PARQUET_DIR.glob(
        "year=*/**/*.parquet"
    )
)

nifty_files = list(
    NIFTY50_DIR.glob(
        "year=*/**/*.parquet"
    )
)

if stock_files:

    stock_glob = str(
        PARQUET_DIR / "**" / "*.parquet"
    )

    print("STOCK DATA SUMMARY")

    stock_summary = con.execute(
        f'''
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT symbol) AS symbols,
            MIN(date) AS min_date,
            MAX(date) AS max_date
        FROM read_parquet(
            '{stock_glob}',
            hive_partitioning=true
        )
        '''
    ).df()

    display(stock_summary)

else:
    print("No stock Parquet files found.")


if nifty_files:

    nifty_glob = str(
        NIFTY50_DIR / "**" / "*.parquet"
    )

    print("NIFTY 50 SUMMARY")

    nifty_summary = con.execute(
        f'''
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT symbol) AS symbols,
            MIN(date) AS min_date,
            MAX(date) AS max_date,
            MIN(close) AS min_close,
            MAX(close) AS max_close
        FROM read_parquet(
            '{nifty_glob}',
            hive_partitioning=true
        )
        '''
    ).df()

    display(nifty_summary)

    print("Latest NIFTY 50 rows:")

    latest_nifty = con.execute(
        f'''
        SELECT
            date,
            symbol,
            open,
            high,
            low,
            close,
            volume
        FROM read_parquet(
            '{nifty_glob}',
            hive_partitioning=true
        )
        ORDER BY date DESC
        LIMIT 10
        '''
    ).df()

    display(latest_nifty)

else:
    print("No NIFTY 50 Parquet files found.")

# 21. How to Run

## Full historical sync

Use:

```python
START_DATE = date(2011, 1, 1)
END_DATE = None
```

The notebook is **filesystem-first**.

### Authoritative rule

A date needs downloading **only when its expected Parquet file does not exist**.

```python
stock_missing = [
    day for day in candidates
    if not parquet_path_for_day(day).exists()
]
```

The manifest is **not** used to put dates into or remove dates from this queue.

### Example

If you have:

```text
Weekday candidates : 4,100
Existing Parquet   : 3,332
Missing Parquet    : 768
```

then:

```text
download_queue = 768
```

and the download loop will show:

```text
[1/768]
[25/768]
[50/768]
...
[768/768]
```

It cannot contain the 3,332 existing dates.

## Manifest role

After `stock_missing` is created, the manifest classifies those missing dates:

```text
new
error
not_found
downloaded-but-file-missing
other
```

A previous `error` or `not_found` is retried because the file is still absent.

If the manifest says `downloaded` but the Parquet is absent, the date is also queued because the actual file is missing.

This prevents stale manifest entries from hiding missing data.

## Resume behavior

If Colab stops after downloading 300 of 768 dates:

1. The next run scans the filesystem.
2. Those 300 now-existing Parquets are excluded.
3. Only the remaining missing dates enter `download_queue`.
4. The manifest classifies those remaining dates.
5. Failed dates are retried.

No manifest repair is required.

## Optional deep validation

The normal inventory does NOT open existing stock Parquet files.

For an explicit audit:

```python
DEEP_VALIDATE_EXISTING = True
```

This is intentionally slower.

## Output

```text
/content/drive/MyDrive/quant/data/

├── parquet/
│   └── year=YYYY/
│       └── nse_cm_YYYYMMDD.parquet
├── indices/
│   └── nifty50/
│       └── year=YYYY/
│           └── nifty50_YYYYMMDD.parquet
├── download_manifest.csv
└── indices/
    └── nifty50_manifest.csv
```
